#Powered by [@CoinNoin](https://www.youtube.com/@CoinNoin)
[![Subscribe](https://img.shields.io/badge/YouTube-Subscribe%20@CoinNoin-red?style=for-the-badge&logo=youtube)](https://www.youtube.com/@CoinNoin)

In [ ]:
#@title 1. Setup Environment
#@markdown Installs ComfyUI and required dependencies for Google Colab T4.

import os
import subprocess
from IPython.display import clear_output

WORKSPACE = "/content/ComfyUI"

print("🚀 [@CoinNoin] Initializing ComfyUI Environment...")
if not os.path.exists(WORKSPACE):
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI {WORKSPACE} &> /dev/null
    print("   ✓ ComfyUI Cloned")
else:
    !cd {WORKSPACE} && git pull &> /dev/null
    print("   ✓ ComfyUI Updated")

!apt-get -y install -qq aria2 &> /dev/null
!pip install -q uv
!cd {WORKSPACE} && uv pip install --system -q -r requirements.txt
!uv pip install --system -q xformers!=0.0.18 --extra-index-url https://download.pytorch.org/whl/cu121

clear_output()
print("✅ [@CoinNoin] Environment Ready!")

In [ ]:
#@title 2. Download Models
#@markdown Select either **x2** or **x4** upscale weights.

SCALE_PAIR = "x2 (relayvsr_x2_lightx2v_int8_convrot + flash_vae_x2_fp32)" #@param ["x2 (relayvsr_x2_lightx2v_int8_convrot + flash_vae_x2_fp32)", "x4 (relayvsr_x4_int8_convrot + flash_vae_x4_fp32)"]

import os
import subprocess
import urllib.parse

WORKSPACE = "/content/ComfyUI"
UNET_DIR = os.path.join(WORKSPACE, "models/diffusion_models")
VAE_DIR = os.path.join(WORKSPACE, "models/vae")
INPUT_DIR = os.path.join(WORKSPACE, "input")

os.makedirs(UNET_DIR, exist_ok=True)
os.makedirs(VAE_DIR, exist_ok=True)
os.makedirs(INPUT_DIR, exist_ok=True)

if SCALE_PAIR.startswith("x2"):
    UNET_URL = "https://huggingface.co/Comfy-Org/RelayVSR/resolve/main/diffusion_models/relayvsr_x2_lightx2v_int8_convrot.safetensors"
    VAE_URL = "https://huggingface.co/Comfy-Org/RelayVSR/resolve/main/vae/relayvsr_flash_vae_x2_fp32.safetensors"
else:
    UNET_URL = "https://huggingface.co/Comfy-Org/RelayVSR/resolve/main/diffusion_models/relayvsr_x4_int8_convrot.safetensors"
    VAE_URL = "https://huggingface.co/Comfy-Org/RelayVSR/resolve/main/vae/relayvsr_flash_vae_x4_fp32.safetensors"

def aria_download(url, target_dir, filename=None):
    if not filename:
        filename = os.path.basename(urllib.parse.urlparse(url).path)
    out_path = os.path.join(target_dir, filename)
    if os.path.exists(out_path) and os.path.getsize(out_path) > 100000:
        print(f"   ✅ Cached: {filename}")
        return filename
    print(f"   📥 Downloading: {filename}...")
    subprocess.run([
        "aria2c", "--console-log-level=error", "--summary-interval=10",
        "-c", "-x", "16", "-s", "16", "-k", "1M",
        "-o", filename, url, "-d", target_dir
    ], check=True)
    print(f"   ✅ Saved: {out_path}")
    return filename

ACTIVE_RELAY_UNET = aria_download(UNET_URL, UNET_DIR)
ACTIVE_RELAY_VAE = aria_download(VAE_URL, VAE_DIR)

print("\n✅ [@CoinNoin] Models Ready!")

In [ ]:
#@title 3. Upscale Video
#@markdown ### Easy Guide:
#@markdown 1. Keep **UPLOAD_NEW_VIDEO** checked and run this cell to upload your video (`.mp4`, `.mov`, `.mkv`, `.webm`).
#@markdown 2. To re-run the same uploaded video with new settings, uncheck **UPLOAD_NEW_VIDEO** — it will auto-detect your last video.
#@markdown 3. **Out of Memory (OOM) on T4?** Check **ENABLE_SLICE** and set **DURATION** to `5.0` seconds to process in shorter clips.
#@markdown 4. Keep `STEPS = 1`, `CFG = 1.0`, `euler`, and `simple` for best results.

# --- Input Video ---
UPLOAD_NEW_VIDEO = True #@param {type:"boolean"}

# --- Optional Video Slicing (Use if T4 runs out of VRAM) ---
ENABLE_SLICE = False #@param {type:"boolean"}
START_TIME = 0.0 #@param {type:"number"}
DURATION = 5.0 #@param {type:"number"}
STRICT_DURATION = False #@param {type:"boolean"}

# --- RelayVSR Settings (Workflow Defaults) ---
STEPS = 1 #@param {type:"integer"}
CFG = 1.0 #@param {type:"number"}
SEED = 42 #@param {type:"integer"}
SAMPLER_NAME = "euler" #@param ["euler", "euler_ancestral", "dpmpp_2m", "uni_pc"] {allow-input: true}
SCHEDULER = "simple" #@param ["simple", "normal", "karras", "beta"] {allow-input: true}
VSR_DETAIL = 1 #@param {type:"integer"}

# --- Output Video Settings ---
BIT_DEPTH = "auto" #@param ["auto", "8-bit", "16-bit"] {allow-input: true}
COLOR_SPACE = "sRGB" #@param ["sRGB", "linear", "rec709"] {allow-input: true}
CHROMA_SUBSAMPLING = "auto" #@param ["auto", "4:2:0", "4:2:2", "4:4:4"] {allow-input: true}
SAVE_FORMAT = "auto" #@param ["auto", "mp4", "webm"] {allow-input: true}
SAVE_CODEC = "auto" #@param ["auto", "h264", "vp9", "av1"] {allow-input: true}
AUTO_DOWNLOAD = True #@param {type:"boolean"}

import sys
import os
import json
import time
import shutil
import subprocess
import urllib.request
from google.colab import files
from IPython.display import Video, display, clear_output

WORKSPACE = "/content/ComfyUI"
INPUT_DIR = os.path.join(WORKSPACE, "input")
OUTPUT_DIR = os.path.join(WORKSPACE, "output")
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.chdir(WORKSPACE)

VIDEO_EXTS = (".mp4", ".mov", ".mkv", ".webm", ".avi", ".m4v")

def get_latest_input_video():
    vids = [
        os.path.join(INPUT_DIR, f)
        for f in os.listdir(INPUT_DIR)
        if f.lower().endswith(VIDEO_EXTS)
    ]
    if not vids:
        return None
    latest = max(vids, key=os.path.getmtime)
    return os.path.basename(latest)

def upload_video_to_input():
    print("📤 [@CoinNoin] Select your video file to upload...")
    uploaded = files.upload()
    if not uploaded:
        return None
    up_name = list(uploaded.keys())[0]
    clean_name = os.path.basename(up_name)
    src_path = os.path.join(os.getcwd(), up_name)
    dst_path = os.path.join(INPUT_DIR, clean_name)
    if os.path.exists(src_path):
        shutil.move(src_path, dst_path)
    else:
        with open(dst_path, "wb") as f:
            f.write(uploaded[up_name])
    print(f"   ✅ Uploaded & saved: {clean_name}")
    return clean_name

# Dynamically resolve VIDEO_FILENAME
VIDEO_FILENAME = None
if UPLOAD_NEW_VIDEO:
    VIDEO_FILENAME = upload_video_to_input()

if not VIDEO_FILENAME:
    last_vid = globals().get("LAST_VIDEO_FILENAME")
    if last_vid and os.path.exists(os.path.join(INPUT_DIR, last_vid)):
        VIDEO_FILENAME = last_vid
    else:
        VIDEO_FILENAME = get_latest_input_video()

if not VIDEO_FILENAME:
    print("⚠️ No video found in input folder. Please upload a video now.")
    VIDEO_FILENAME = upload_video_to_input()

if not VIDEO_FILENAME:
    raise RuntimeError("❌ No video uploaded. Please re-run this cell and select a video file.")

LAST_VIDEO_FILENAME = VIDEO_FILENAME

unet_name = globals().get("ACTIVE_RELAY_UNET", "relayvsr_x2_lightx2v_int8_convrot.safetensors")
vae_name = globals().get("ACTIVE_RELAY_VAE", "relayvsr_flash_vae_x2_fp32.safetensors")

print("🔌 [@CoinNoin] Checking ComfyUI Server...")
def ensure_comfyui():
    req = urllib.request.Request("http://127.0.0.1:8188/system_stats")
    try:
        urllib.request.urlopen(req, timeout=3)
        print("   🟢 ComfyUI server is already running.")
    except Exception:
        print("   🚀 Starting ComfyUI server with --lowvram for T4...")
        subprocess.Popen(
            [sys.executable, "main.py", "--port", "8188", "--lowvram"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )
        for _ in range(90):
            try:
                urllib.request.urlopen(req, timeout=3)
                print("   🟢 ComfyUI server is ready!")
                break
            except Exception:
                time.sleep(2)
        else:
            raise RuntimeError("ComfyUI server failed to start within timeout.")

ensure_comfyui()
clear_output()

print(f"🎞️ Active Input Video: {VIDEO_FILENAME}")
print(f"⚙️ Models: {unet_name} + {vae_name}")

# Build API prompt matching utility_relayvsr_video_x2_upscale.json + subgraph
api_prompt = {
    "1": {
        "class_type": "LoadVideo",
        "inputs": {
            "file": VIDEO_FILENAME
        }
    },
    "4": {
        "class_type": "UNETLoader",
        "inputs": {
            "unet_name": unet_name,
            "weight_dtype": "default"
        }
    },
    "7": {
        "class_type": "VAELoader",
        "inputs": {
            "vae_name": vae_name
        }
    }
}

video_source_link = ["1", 0]

if ENABLE_SLICE:
    api_prompt["202"] = {
        "class_type": "Video Slice",
        "inputs": {
            "video": video_source_link,
            "start_time": START_TIME,
            "duration": DURATION,
            "strict_duration": STRICT_DURATION
        }
    }
    video_source_link = ["202", 0]

api_prompt["2"] = {
    "class_type": "GetVideoComponents",
    "inputs": {
        "video": video_source_link
    }
}

api_prompt["5"] = {
    "class_type": "RelayVSRConditioning",
    "inputs": {
        "model": ["4", 0],
        "images": ["2", 0]
    }
}

api_prompt["6"] = {
    "class_type": "KSampler",
    "inputs": {
        "seed": SEED,
        "steps": STEPS,
        "cfg": CFG,
        "sampler_name": SAMPLER_NAME,
        "scheduler": SCHEDULER,
        "denoise": 1.0,
        "model": ["4", 0],
        "positive": ["5", 0],
        "negative": ["5", 0],
        "latent_image": ["5", 1]
    }
}

api_prompt["104"] = {
    "class_type": "VSRDecode",
    "inputs": {
        "vae": ["7", 0],
        "samples": ["6", 0],
        "images": ["2", 0],
        "detail": VSR_DETAIL
    }
}

api_prompt["105"] = {
    "class_type": "CreateVideo",
    "inputs": {
        "images": ["104", 0],
        "audio": ["2", 1],
        "fps": ["2", 2],
        "bit_depth": BIT_DEPTH,
        "color_space": COLOR_SPACE,
        "chroma_subsampling": CHROMA_SUBSAMPLING,
        "codec": "none"
    }
}

api_prompt["106"] = {
    "class_type": "SaveVideo",
    "inputs": {
        "video": ["105", 0],
        "filename_prefix": "video/RelayVSR",
        "format": SAVE_FORMAT,
        "codec": SAVE_CODEC
    }
}

payload = json.dumps({"prompt": api_prompt}).encode("utf-8")
req = urllib.request.Request(
    "http://127.0.0.1:8188/prompt",
    data=payload,
    headers={"Content-Type": "application/json"}
)

print(f"\n📥 [@CoinNoin] Submitting RelayVSR job...")
try:
    with urllib.request.urlopen(req) as resp:
        prompt_id = json.loads(resp.read().decode("utf-8"))["prompt_id"]
except urllib.error.HTTPError as e:
    print(f"❌ ComfyUI API Error: {e.read().decode('utf-8')}")
    raise

print("🎬 Upscaling video frames (this may take a few minutes on T4)...")
outputs = {}
while True:
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:8188/history/{prompt_id}") as h_resp:
            hist = json.loads(h_resp.read().decode("utf-8"))
        if prompt_id in hist:
            outputs = hist[prompt_id].get("outputs", {})
            break
    except Exception:
        pass
    time.sleep(2)

saved_videos = []
for node_id, node_out in outputs.items():
    for key in ("videos", "gifs", "images"):
        if key in node_out:
            for item in node_out[key]:
                fname = item.get("filename")
                subf = item.get("subfolder", "")
                if fname:
                    vpath = os.path.join(OUTPUT_DIR, subf, fname) if subf else os.path.join(OUTPUT_DIR, fname)
                    if os.path.exists(vpath):
                        saved_videos.append(vpath)
                        print(f"\033[92m✓ [@CoinNoin] Saved Output: {vpath}\033[0m")
                        display(Video(vpath, embed=True))

if AUTO_DOWNLOAD and saved_videos:
    print("\n📥 [@CoinNoin] Downloading output video(s)...")
    for vp in saved_videos:
        files.download(vp)